# 1.2.8 Web Scraping & APIs

Fetch data from public APIs using stdlib `urllib`: weather, REST, HuggingFace, pagination, caching.

In [ ]:
import urllib.request, urllib.parse, urllib.error, json, time
from pathlib import Path
DATA=Path('data'); DATA.mkdir(exist_ok=True)

def http_get_json(url, params=None):
    if params: url=url+'?'+urllib.parse.urlencode(params)
    try:
        req=urllib.request.Request(url,headers={'User-Agent':'ML-Learning/1.0'})
        with urllib.request.urlopen(req,timeout=10) as r: return json.loads(r.read())
    except Exception as e: print(f'Error: {e}'); return None

print('Helper ready')

In [ ]:
# Weather API (Open-Meteo — free, no key)
data=http_get_json('https://api.open-meteo.com/v1/forecast',
    {'latitude':51.5,'longitude':-0.12,'daily':'temperature_2m_max','forecast_days':3,'timezone':'UTC'})
if data:
    for d,t in zip(data['daily']['time'],data['daily']['temperature_2m_max']):
        print(f'{d}: {t}°C')
else:
    print('Offline — no data')

In [ ]:
# REST API pagination (JSONPlaceholder)
posts=[]
for page in [1,2]:
    data=http_get_json('https://jsonplaceholder.typicode.com/posts',{'_page':page,'_limit':3})
    if data: posts.extend(data)
    time.sleep(0.2)
print(f'Fetched {len(posts)} posts')
for p in posts[:3]: print(f"  id={p['id']} title={p['title'][:40]}")

In [ ]:
# Download a dataset + cache
dest=DATA/'titanic.csv'
if not dest.exists():
    raw=urllib.request.urlopen('https://huggingface.co/datasets/phihung/titanic/resolve/main/train.csv').read()
    dest.write_bytes(raw)
print(f'Dataset: {dest.name} ({dest.stat().st_size:,} bytes)')

In [ ]:
# Disk cache
cache_file=DATA/'api_cache.json'
cache=json.loads(cache_file.read_text()) if cache_file.exists() else {}
def cached_get(url,key):
    if key in cache: print(f'HIT: {key}'); return cache[key]
    data=http_get_json(url)
    if data: cache[key]=data; cache_file.write_text(json.dumps(cache))
    print(f'MISS: {key}'); return data
cached_get('https://jsonplaceholder.typicode.com/todos/1','todo_1')
cached_get('https://jsonplaceholder.typicode.com/todos/1','todo_1')